In [ ]:
# --- INSTALACIÓN DE DEPENDENCIAS ---
# Instalamos la versión oficial y estable de Ultralytics (que incluye RT-DETR)
!pip install -U ultralytics
!pip install kagglehub

In [ ]:
# --- DESCARGA DEL DATASET RES-Dataset ---
print(" Descargando RES-Dataset de Kaggle...")
dataset_path = kagglehub.dataset_download("wxqwxq321/res-dataset")

print(f" Dataset descargado en: {dataset_path}")

# --- BÚSQUEDA AUTOMÁTICA DEL YAML ---
yaml_path = None
for root, dirs, files in os.walk(dataset_path):
    if 'data.yaml' in files:
        yaml_path = os.path.join(root, 'data.yaml')
        break

if yaml_path:
    print(f" Archivo de configuración encontrado en: {yaml_path}")
else:
    print(" ERROR: No se encontró 'data.yaml'.")

In [ ]:
# --- ENTRENAMIENTO CON RT-DETR (Transformers) ---
from ultralytics import RTDETR  # Importamos la clase del Transformer

# 1. Cargar el modelo base RT-DETR-L.
print(" Cargando modelo RT-DETR-L (Atención Global)...")
model = RTDETR('rtdetr-l.pt')

# 2. Configurar Hiperparámetros y Entrenar
print(" Iniciando entrenamiento táctico de RT-DETR con RES-Dataset...")

results = model.train(
    data=yaml_path,
    epochs=50,
    imgsz=640,
    batch=4,
    project='runs/detect',
    name='rtdetr_res_dataset',
    exist_ok=True,
    plots=True,

    # --- DATA AUGMENTATION PARA TRANSFORMERS ---
    mosaic=1.0,
    mixup=0.15,
    scale=0.5,
    degrees=15.0,

    # --- PREVENCIÓN DE DEFORMACIONES FÍSICAS IMPOSIBLES ---
    shear=0.0,        # Desactivado para no deformar físicamente los objetos
    perspective=0.0   # Desactivado para mantener las proporciones realistas top-down
)

print(" Entrenamiento con Transformers finalizado.")

In [ ]:
import shutil
import os
from google.colab import drive

# --- 1. VERIFICACIÓN Y MONTAJE DE GOOGLE DRIVE ---
def asegurar_drive():
    """Comprueba si Drive está montado. Si no, lanza el proceso de autenticación."""
    if not os.path.ismount('/content/drive'):
        print(" Google Drive no está montado. Iniciando proceso de conexión...")
        drive.mount('/content/drive')
    else:
        print(" Google Drive ya se encuentra montado y operativo.")

# Ejecutamos la comprobación de seguridad
asegurar_drive()


# --- 2. EXTRACCIÓN Y GUARDADO DEL MODELO ---
print("\n Iniciando protocolo de guardado en la nube...")

ruta_origen = 'runs/detect/rtdetr_res_dataset/weights/best.pt'

carpeta_drive = '/content/drive/MyDrive/Modelos_TFG/'

os.makedirs(carpeta_drive, exist_ok=True)

ruta_destino = os.path.join(carpeta_drive, 'rtdetr_res_best.pt')

if os.path.exists(ruta_origen):
    shutil.copy(ruta_origen, ruta_destino)
    print(f" ¡MISIÓN CUMPLIDA! El modelo RT-DETR ha sido asegurado con éxito:")
    print(f"   -> Ubicación: {ruta_destino}")
else:
    print(f" ALERTA: No se encontró el modelo en {ruta_origen}.")